## This file exists as a stand-alone runnable script, such that you can employ it's function, either against a single workspace, or against many, i.e. in a loop

In [ ]:
%run fabric-notebook-config

In [ ]:
# asserted column names, as the shortcut from Azure Data lake doesnt support underscores `_` in colnames, but the APIs return with spaces ` `
# this assertion normalises the `target state` and API return `current state` tables
workspace_access_column_names = ["workspace", "Email Address", "Group User Access Right", "Identifier", "Graph Id", "Principal Type"]
workspace_access_filename = 'workspace-membership-intended.csv'
path = f"{lakehouse_path}/{workspace_access_filename}"

In [ ]:
# Load data into pandas DataFrame
workspace_members_spark = spark.read.format("csv").option('header', 'true').load(path)
workspace_membership_intended = workspace_members_spark.toPandas()
# data cleaning
workspace_membership_intended = workspace_membership_intended.replace(np.nan, None)
workspace_membership_intended.drop('User_Name', axis=1, inplace=True) # drop this so if the group is renamed we don't have to worry about it
workspace_membership_intended.columns = workspace_access_column_names

display(workspace_membership_intended)

In [ ]:
# get a unique list of workspaces to iterate over
workspaces = set(workspace_membership_intended['workspace'])

In [ ]:
def enforce_workspace_membership(workspace_id):
    """
    Compares the current state of workspace membership with the target state (both in pandas data-frame format) and performs necessary updates (adding or removing users/groups) to align the current state with the target state.
    Requires active Fabric administrator rights, as labs.admin.delete_user_from_workspace / labs.admin.add_user_to_workspace don't currently support SP based auth (Github issue: https://github.com/microsoft/semantic-link-labs/issues/1104)

    Parameters
    ----------
    - workspace_id: unique ID for the workspace, available in the URL, or via plugins / APIs

    Returns
    -------
    None
    This function does not return any value. It adds & removes users/groups/service principals from the workspace based on the differences between `current_state_df` and `target_state_df`.

    Notes
    -----
    - If there are no differences between the current state and the target state, no operations are performed.
    - If differences exist:
    - Users/groups to be added are identified and processed.
    - Users/groups to be removed are identified and processed.
    - For removing groups, a flag is passed to distinguish them from service principals, as otherwise the underlying Fabric API throws an error.

    - Is sensative to group name changes

    Internally generates
        
    - current_state_df : pandas.DataFrame
        A DataFrame representing the current state of workspace membership. It is generated by the Fabric Semantic-Link Labs Admin package function `list_workspace_users()` 
    - target_state_df : pandas.DataFrame
        A DataFrame representing the desired target state of workspace membership. It' structure should match that which is returned by Semantic-Link Labs Admin package function `list_workspace_users()`. This is what the workspace access policies will be updated to match.


    Examples
    --------
    >>> import pandas as pd
    >>> current_state_df = pd.DataFrame({
        'Identifier': ['user1', 'user2', 'group1'],
        Principal Type': ['User', 'User', 'Group'],
        Group User Access Right': ['Admin', 'Member', 'Contributor'],
        workspace': ['Workspace1', 'Workspace1', 'Workspace1']
        })
    >>> target_state_df = pd.DataFrame({
        'Identifier': ['user1', 'user3', 'group1'],
        'Principal Type': ['User', 'User', 'Group'],
        'Group User Access Right': ['Admin', 'Member', 'Contributor'],
        'workspace': ['Workspace1', 'Workspace1', 'Workspace1']
        })
    >>> enforce_workspace_membership(current_state_df, target_state_df)
    some differences
    start removing users and principals
    removed `user2`
    removed all users and principals
    removing groups
    removed all groups
    end remove logic
    start adding users, groups and principals
    added `user3`
    end add logic

    # Description generated partially by QChat
    """

    # get desired state
    target_state_df = workspace_membership_intended[workspace_membership_intended['workspace'] == workspace_id]
    # get current state - with error handling
    ## check the workspace exists & is not in temp deletion hold
    ## This API funciton returns a 0 row dataframe if not found (not an error), so can safely call it outside the try-catch
    ### Account for not using SP context based on library variable value
    if orchestration_sp_app_object_id == executing_user: 
        # Use SP context - requires Keyvault access to read secrets(PIM or via SP caller)
        with labs.service_principal_authentication(
            key_vault_uri = key_vault_uri,
            key_vault_tenant_id = 'tenant-fabric-sp',
            key_vault_client_id = 'fabric-admin-sp-app-id', 
            key_vault_client_secret = admin_sp_client_secret_name[1]
            ):
            workspace_details = labs.admin.list_workspaces(workspace = workspace_id)
            try: 
                    current_state_df = labs.admin.list_workspace_users(workspace_id)
            except: 
                manage_workspace_members_invalid_dict = {
                    "message": "Target workspace is does not exist",
                    "invalid_workspace": workspace_id
                }   
                mssparkutils.notebook.exit(manage_workspace_members_invalid_dict)
        ##### End SP context
    else:
        # use User context (requires Fabric Admin)
        workspace_details = labs.admin.list_workspaces(workspace = workspace_id)
        try: 
            current_state_df = labs.admin.list_workspace_users(workspace_id)
        except: 
            manage_workspace_members_invalid_dict = {
                "message": "Target workspace is does not exist",
                "invalid_workspace": workspace_id
            }   
            mssparkutils.notebook.exit(manage_workspace_members_invalid_dict)
        ##### End user context

    ## now we know it exits, has it been deleted? If so, halt and prompt to restore / remove from list 
    if workspace_details['State'].iloc[0] == 'Deleted':
        manage_workspace_members_invalid_dict = {
                "message": "Target workspace has been deleted, remove it from the mapping, or restore it",
                "invalid_workspace": workspace_id
        }   
        mssparkutils.notebook.exit(manage_workspace_members_invalid_dict)
    # normal the structure between API return and target state df
    current_state_df.insert(0, 'workspace', workspace_id)
    # drop the user name column in-case this throws issues - esp w/ group renames
    current_state_df.drop('User Name', axis=1, inplace=True)
    current_state_df.sort_values(workspace_access_column_names, inplace= True)
    target_state_df = target_state_df.sort_values(by = workspace_access_column_names)

    '''
    ------------------- READ THIS COMMENT --------------------------
    
    Because there is currently no Service Principal support for the Admin APIS `labs.admin.add_user_to_workspace()` and `labs.admin.delete_user_from_workspace()`, we are forced to do this component outside of SP contexts.
    Have raised a feature enhancement request on SLL github: https://github.com/microsoft/semantic-link-labs/issues/1104

    If the SP context for AddUser / DeleteUser (from Workspace) Admin APIs is added, we can then indent this entire function under 1x `with labs.service_principal_authentication(...)` block
    
    -------------- THANK YOU FOR YOUR ATTENTION --------------------

    '''

    print("Checking Workspace: " + target_state_df['workspace'].iloc[0])
    ## Check if there's variation between the target state and the actual state
    if target_state_df.reset_index(drop=True).equals(current_state_df.reset_index(drop=True)):
        print("No differences")
    else:
        print("some differences")
        # generate delta between desired and actual
        delta_df = pd.merge(target_state_df, current_state_df, on=['Identifier'], how='outer',indicator=True)
        # split the adds and removes into their own little DFs
        to_remove = delta_df[delta_df['_merge'] == "right_only"]
        to_add = delta_df[delta_df['_merge'] == "left_only"]
        # Identify what permissions need to be updated
        both = delta_df[delta_df['_merge'] == "both"]
        to_update_within = both[both['Group User Access Right_x'] != both['Group User Access Right_y']]

        # get the list of users to add to the workspace in a neat 7 tidy format
        to_add = to_add.iloc[:, 0:6]
        to_add.columns = workspace_access_column_names
        # get the list of users to remove from the workspace in a neat 7 tidy format
        temp = to_remove.iloc[:,6:11]
        temp.insert(4, 'Identifier', to_remove['Identifier'])
        to_remove = temp
        to_remove.columns = workspace_access_column_names

        ## start processing changes
        # do we need to modify anyone's perms?
        if pd.DataFrame(to_update_within).size == 0: 
            print("no one to adjust")
        else:
            print('start modifying users, groups and principals')
            for index, row in to_update_within.iterrows():
                # need to check what type of person we're adding, as needs to be email for ppl and Entra Object ID for users
                if row['Principal Type_x'] == "User":
                    labs.admin.add_user_to_workspace(
                    row['Email Address_x'], # Email address is alias for UPN, Graph Object ID throws errors
                    row['Group User Access Right_x'],
                    row['Principal Type_x'],
                    row['workspace_x']
                    )
                else:
                    labs.admin.add_user_to_workspace(
                    row['Identifier'], # Graph Object ID throws for SPs and Groups
                    row['Group User Access Right_x'],
                    row['Principal Type_x'],
                    row['workspace_x']
                    )
            print('end modification logic')

        # do we need to remove anyone? 
        if pd.DataFrame(to_remove).size == 0:
            print("no removals needed")
        else:
            # logic to go here
            users_sp_to_rm = to_remove[to_remove['Principal Type'] != "Group"]
            groups_to_rm = to_remove[to_remove['Principal Type'] == "Group"]
            print("start removing users and principals")
            for index, row in users_sp_to_rm.iterrows():
                labs.admin.delete_user_from_workspace(
                row['Identifier'], 
                row['workspace']
                )
            print("removed all users and principals")
            print("removing groups")
            for index, row in groups_to_rm.iterrows():
                labs.admin.delete_user_from_workspace(
                row['Identifier'],
                row['workspace']
                # have to add a flag when removing groups, otherwise it think's it's the ID of an SP: # https://learn.microsoft.com/en-au/rest/api/power-bi/admin/groups-delete-user-as-admin
                , True   
                )
            print ("removed all groups")
        # do we need to add anyone?
        if pd.DataFrame(to_add).size == 0: 
            print("no one to add")
        else:
            print('start adding users, groups and principals')
            for index, row in to_add.iterrows():
                # need to check what type of person we're adding, as needs to be email for ppl and Entra Object ID for users
                if row['Principal Type'] == "User":
                    labs.admin.add_user_to_workspace(
                    user = row['Email Address'], # Email address is alias for UPN, Graph Object ID throws errors
                    role = row['Group User Access Right'],
                    principal_type  = row['Principal Type'],
                    workspace  = row['workspace']
                    )
                else:
                    labs.admin.add_user_to_workspace(
                    user = row['Identifier'], # Graph Object ID for SPs and Groups
                    role = row['Group User Access Right'],
                    principal_type = row['Principal Type'],
                    workspace = row['workspace']
                    )
            print("done adding users, groups and principals")
    print('finished with workspace: '+ target_state_df['workspace'].iloc[0])